# Homework #2 - Basic Statistical Tests

By Tyler Engle for EE 499: Wearable Informatics

### Background:
Wearable devices have seen great utility in behavioral health science. Though they also provide
very different data from what is typically collected in such fields, many of the behavioral health
statistical methods apply to data from these sources too.
### Assignment:
Your assignment will be to write several functions to perform statistical tests on a collection of data
sets and then use those functions to evaluate the dataset given. You may use Octave/MATLAB,
Python, R, or any of these in Notebooks, but you must write the functions to compute and perform
the statistics outlined below from scratch. You shouldn’t need any libraries beyond basics for
reading CSVs and two statistic value lookups. You can also use pandas, but I don’t want you
seeking out ActiGraph specific or statistics specific libraries for what you should be writing from
scratch.

In [1]:
# Import the libraries needed to run i.e. Pandas, math, and scipy
import pandas as pd
import math
from scipy import stats
import glob
import os

## Functions
### Harmonic Means 
Your function should accept N complete datasets (vectors / arrays / dataframes / your choice) and
return the harmonic mean of each. 

In [2]:
## Calculates either the arithmetic or harmonic mean of a given dataset.
def CalculateHarmonicMean(data, mean_type='arithmetic'):
    n = len(data)
    
    if mean_type == 'harmonic':
        sumOfReciprocals = sum(1 / x for x in data)
        return n / sumOfReciprocals      
    elif mean_type == 'arithmetic':
        return sum(data) / n      
    else:
        raise ValueError("mean_type must be either 'arithmetic' or 'harmonic'")

### Pooled Stardard Deviation
Your function should be able to accept the standard deviation and number of samples from each
sample set that is to be combined. It should also be able to self identify the k, and should accept
any number of pairings (calculated σ and given n).
Because I want you to build your stats and programming knowledge, you should also write a basic
standard deviation function. Here’s a reminder of the math needed to calculate the standard
deviation of a sample.

In [3]:
## Calculates the standard deviation of a sample using the formula provided.
def CalculateSTD(data):
    mu = CalculateHarmonicMean(data, mean_type='arithmetic')
    n = len(data)
    sumSquaredDiff = sum((x -mu)**2 for x in data)
    variance = (1 / n) * sumSquaredDiff
    return math.sqrt(variance)

## Calculate the pooled standard deviation.
def PooledSTD(*sampleStats):
    k = len(sampleStats)
    numerator = 0
    denominatorSumN = 0
    
    for std, n in sampleStats:
        numerator += (n - 1) * (std**2)
        denominatorSumN += n
    denominator = denominatorSumN - k
    
    return math.sqrt(numerator / denominator)

### T-Test
Your t-test function should accept both of the following:
1. two data sets, from which all parameters needed to calculate a t− value can be calculated
2. µ, σ, and n for each of the two data sets, from which the t− value can be calculated
and should return a p− value for the given datasets (or measures of the datasets).

This function is an exception to the requirement that you not use any special libraries. If you are
using MATLAB or Octave, you may use the following to translate your t− value into and return
a p− value:
tcdf(t, df)
And if you are using Python:
from scipy import stats
stats.t.cdf
Or if you have elected to use R:
pt()
Consider how you might change this function to optionally make use of your harmonic mean func-
tion. Can you provide an input flag that allows you to specify to use either a basic arithmetic mean
or your harmonic mean? How about a default of one or the other?

In [4]:
# Performs a t-test. Accepts either two raw datasets OR the pre-calculated statistics (mu, sigma, n) for two datasets.
def CustomTTest(data1=None, data2=None, stats1=None, stats2=None):
    
    if data1 is not None and data2 is not None:
        n1, n2 = len(data1), len(data2)
        mu1, mu2 = CalculateHarmonicMean(data1), CalculateHarmonicMean(data2)
        std1, std2 = CalculateSTD(data1), CalculateSTD(data2)
    elif stats1 is not None and stats is not None:
        mu1, std1, n1 = stats1
        mu2, std2, n2 = stats2
    else:
        raise ValueError("You must provide either two datasets (data1, data2) OR two sets of stats (stats1, stats2).")

    df = n1 + n2 - 2
    variance1 = std1**2
    variance2 = std2**2
    sp = math.sqrt(((n1 - 1) * variance1 + (n2 - 1) * variance2) / df)
    tNumerator = mu1 - mu2
    tDenominator = sp * math.sqrt((1 / n1) + (1 / n2))
    tStat = tNumerator / tDenominator
    pValue = 2 * (1 - stats.t.cdf(abs(tStat), df))
    
    return tStat, pValue

### ANOVA
Your ANOVA function should accept three or more datasets and calculate the F− stat, but then
should use the equivalent to tcdf() to find the p− value. This function is also an exception, in that
you can use a library to find the p− value from the F− stat, but like above, you can not use a
library to calculate the F− stat.
Think carefully about how you can accept some undefined number of datasets and still perform the
calculation.
See the slides and Wikipedia for details on how to perform the math.

In [5]:
# Performs an ANOVA test on three or more datasets.
def CustomANOVA(*groups):
    m = len(groups)
    allObservations = [item for group in groups for item in group]
    N = len(allObservations)
    overallMean = CalculateHarmonicMean(allObservations)
    ssTotal = sum((x - overallMean)**2 for x in allObservations)
    ssBetween = 0
    
    for group in groups:
        nJ = len(group)
        groupMean = CalculateHarmonicMean(group)
        ssBetween += nJ * (groupMean - overallMean)**2
        
    ssWithin = ssTotal - ssBetween
    dfBetween = m - 1
    dfWithin = N - m
    msBetween = ssBetween / dfBetween 
    msWithin = ssWithin / dfWithin
    fStat = msBetween / msWithin
    pValue = 1 - stats.f.cdf(fStat, dfBetween, dfWithin)
    
    return fStat, pValue

### Repeated Measures ANOVA
Finally, you need to implement a repeated measures ANOVA, often referred to as rANOVA or
RMANOVA. See Wikipedia for details on the math.

In [6]:
# Performs a Repeated Measures ANOVA (rANOVA).
def RepeatedMeasuresANOVA(*conditions):
    k = len(conditions)
    n = len(conditions[0])
    allData = [item for condition in conditions for item in condition]
    overallMean = CalculateHarmonicMean(allData)
    conditionMeans = [CalculateHarmonicMean(c) for c in conditions]
    
    subjectMeans = []
    for subjectIndex in range(n):
        subjectData = [conditions[conditionIndex][subjectIndex] for conditionIndex in range(k)]
        subjectMeans.append(CalculateHarmonicMean(subjectData))
    
    ssSubjects = 0
    for i in range(n):
        for j in range(k):
            ssSubjects += (conditions[j][i] - subjectMeans[i])**2

    ssConditions = 0
    for j in range(k):
        ssConditions += n * (conditionMeans[j] - overallMean)**2

    ssError = ssSubjects - ssConditions
    dfConditions = k - 1
    dfSubjects = n - 1
    dfError = dfConditions * dfSubjects
    msConditions = ssConditions / dfConditions
    msError = ssError / dfError
    fStat = msConditions / msError
    pValue = 1 - stats.f.cdf(fStat, dfConditions, dfError)
    
    return fStat, pValue

## Applications of Your Functions 
Using the functions you have developed from above, and the data provided in the git repository
(‘Sample Data’ directory), compute the following and answer the questions posed. The first four
questions should make use of the concurrently measured data sets (‘actigraph and fitbit’ directory),
while the last question makes use of the Fitbit only data set (‘multiyear’ directory).
### Daily Steps 
Pick one of the devices: How many steps per day, on average do the subjects walk? Use the
harmonic and arithmetic mean. Are they different? Why?

In [7]:
fitbitPath = 'Sample Data/actigraph and fitbit/fitbit/*_FB_minuteSteps.csv'
fitbitFiles = glob.glob(fitbitPath)

if len(fitbitFiles) == 0:
    fitbitFiles = glob.glob('/Sample Data/actigraph and fitbit/fitbit/*_FB_minuteSteps.csv')

if len(fitbitFiles) == 0:
    fitbitFiles = glob.glob('*_FB_minuteSteps.csv')

if len(fitbitFiles) == 0:
    print("ERROR: Still cannot find the Fitbit files.")
else:
    print(f"Found {len(fitbitFiles)} Fitbit step files! Loading data...")
    dfList = []
    for file in fitbitFiles:
        tempDf = pd.read_csv(file)
        subjectId = os.path.basename(file).split('_')[0]
        
        dateColName = [col for col in tempDf.columns if 'Activity' in col and 'Hour' in col][0]
        
        measureCols = [col for col in tempDf.columns if col != dateColName]
        
        tempDf['hourlySteps'] = tempDf[measureCols].sum(axis=1)
        
        tempDf['date'] = pd.to_datetime(tempDf[dateColName]).dt.date
        tempDf['subjectId'] = subjectId
        
        dfList.append(tempDf[['subjectId', 'date', 'hourlySteps']])

    dailyFitbitSteps = pd.concat(dfList, ignore_index=True).groupby(['subjectId', 'date'])['hourlySteps'].sum().reset_index()
    dailyFitbitSteps.rename(columns={'hourlySteps': 'steps'}, inplace=True)
    print("Fitbit data successfully formatted with actual step counts!")

actigraphPath = 'Sample Data/actigraph and fitbit/actigraph/*_AG*.csv'
actigraphFiles = glob.glob(actigraphPath)

if len(actigraphFiles) == 0:
    actigraphFiles = glob.glob('/Sample Data/actigraph and fitbit/actigraph/*_AG*.csv')

if len(actigraphFiles) == 0:
    actigraphFiles = glob.glob('*_AG*.csv')

if len(actigraphFiles) > 0:
    print(f"Found {len(actigraphFiles)} ActiGraph files! Loading data...")
    agList = []
    
    agColumns = ['Axis1', 'Axis2', 'Axis3', 'Steps', 'Lux', 'IncOff', 'IncStand', 'IncSit', 'IncLie']
    
    for file in actigraphFiles:
        tempDf = pd.read_csv(file, skiprows=10, names=agColumns)
        
        subjectId = os.path.basename(file).split('_')[0]
        tempDf['subjectId'] = subjectId
        
        tempDf['dayIndex'] = tempDf.index // 1440
        
        dailySums = tempDf.groupby(['subjectId', 'dayIndex'])['Steps'].sum().reset_index()
        
        dailySums.rename(columns={'Steps': 'steps'}, inplace=True)
        agList.append(dailySums)

    dailyAgSteps = pd.concat(agList, ignore_index=True)
    print("ActiGraph data successfully formatted!")
else:
    print("Could not find ActiGraph files.")

multiyearPath = 'Sample Data/multiyear/dailySteps.csv'
if not os.path.exists(multiyearPath):
    multiyearPath = '/Sample Data/multiyear/dailySteps.csv'

if os.path.exists(multiyearPath):
    print("Found Multiyear dailySteps.csv! Loading data...")
    multiyearDf = pd.read_csv(multiyearPath)
    multiyearDf['date'] = pd.to_datetime(multiyearDf['ActivityDay'])
    multiyearDf['month'] = multiyearDf['date'].dt.month
    multiyearDf.rename(columns={'StepTotal': 'steps'}, inplace=True)
    print("Multiyear data successfully formatted!")
else:
    print(f"Could not find 'dailySteps.csv' at {multiyearPath}.")

def calculateMeans(*datasets, meanType='harmonic'):
    calculatedMeans = []
    for data in datasets:
        n = len(data)
        if n == 0:
            calculatedMeans.append(0)
            continue
            
        if meanType == 'harmonic':
            sumOfReciprocals = sum(1 / x for x in data if x > 0)
            calculatedMeans.append(n / sumOfReciprocals if sumOfReciprocals > 0 else 0)
        elif meanType == 'arithmetic':
            calculatedMeans.append(sum(data) / n)
    return calculatedMeans

subjects = dailyFitbitSteps['subjectId'].unique()
subjectStepLists = []

for subject in subjects:
    steps = dailyFitbitSteps[dailyFitbitSteps['subjectId'] == subject]['steps'].tolist()
    cleanSteps = [s for s in steps if s > 0]
    subjectStepLists.append(cleanSteps)

arithmeticMeans = calculateMeans(*subjectStepLists, meanType='arithmetic')
harmonicMeans = calculateMeans(*subjectStepLists, meanType='harmonic')

print("--- Average Daily Steps (Fitbit) ---")
for i, subject in enumerate(subjects):
    print(f"Subject {subject}:")
    print(f"  Arithmetic Mean: {arithmeticMeans[i]:.2f} steps")
    print(f"  Harmonic Mean:   {harmonicMeans[i]:.2f} steps\n")

Found 4 Fitbit step files! Loading data...
Fitbit data successfully formatted with actual step counts!
Found 8 ActiGraph files! Loading data...
ActiGraph data successfully formatted!
Found Multiyear dailySteps.csv! Loading data...
Multiyear data successfully formatted!
--- Average Daily Steps (Fitbit) ---
Subject 1:
  Arithmetic Mean: 12815.24 steps
  Harmonic Mean:   5152.95 steps

Subject 2:
  Arithmetic Mean: 9672.74 steps
  Harmonic Mean:   8752.93 steps

Subject 3:
  Arithmetic Mean: 10977.09 steps
  Harmonic Mean:   133.66 steps

Subject 4:
  Arithmetic Mean: 9541.44 steps
  Harmonic Mean:   7486.31 steps



/var/folders/bb/t9ccmlt54r719df_6tklz4lw0000gn/T/ipykernel_14090/825170344.py:25: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  tempDf['date'] = pd.to_datetime(tempDf[dateColName]).dt.date
/var/folders/bb/t9ccmlt54r719df_6tklz4lw0000gn/T/ipykernel_14090/825170344.py:25: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  tempDf['date'] = pd.to_datetime(tempDf[dateColName]).dt.date
/var/folders/bb/t9ccmlt54r719df_6tklz4lw0000gn/T/ipykernel_14090/825170344.py:25: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  tempDf['date'] = pd.to_datetime(tempDf[dateColName]).dt.date
/var/folders/

Using the Fitbit data, we calculated both the arithmetic and harmonic means for the subjects' average daily steps and yes, they are significantly different. The harmonic mean is consistently lower than the arithmetic mean for every single subject, and in the case of Subject 3, it is drastically lower.
This difference occurs because of how the two formulas handle variance/outliers. The arithmetic mean calculates a standard average, giving equal weight to all values, making it easily skewed upwards by highly active days.
Because harmonic mean uses reciprocals, the harmonic mean is highly sensitive to very small numbers and heavily discounts large numbers. In behavioral health data, a participant might have a day where they barely wore the Fitbit. As seen with Subject 3, these extremely low-activity "non-wear" days act as heavy mathematical wieghts that pull the harmonic mean down, making it much more conservative than the arithmetic mean.

### Group Variance
Pick one of the devices: What is the variance of the group? (across subjects, pooled standard
deviation)

In [8]:
subjects = dailyFitbitSteps['subjectId'].unique()
subjectStats = []

for subject in subjects:
    steps = dailyFitbitSteps[dailyFitbitSteps['subjectId'] == subject]['steps'].tolist()
    subjectStats.append((CalculateSTD(steps), len(steps)))

groupPooledStd = PooledSTD(*subjectStats)
groupVariance = groupPooledStd ** 2

print(f"Group Pooled Standard Deviation: {groupPooledStd:.2f}")
print(f"Group Variance: {groupVariance:.2f}")

Group Pooled Standard Deviation: 4882.75
Group Variance: 23841222.76


The pooled standard deviation across all subjects for the Fitbit was calculated to be 4882.75 steps, resulting in a biiiggg group variance of 23841222.76. This large variance indicates a great dispersion in the daily step counts, suggests that there are significant changes in physical activity.

### Comparing the Devices 
Use both of the devices: Does the Fitbit report the same step measures as the ActiGraph? (t-test)

In [9]:
fitbitAllSteps = dailyFitbitSteps['steps'].tolist()
actigraphAllSteps = dailyAgSteps['steps'].tolist()

tStat, pValue = CustomTTest(data1=fitbitAllSteps, data2=actigraphAllSteps)

print(f"T-Statistic: {tStat:.4f} | P-Value: {pValue:.4e}")

T-Statistic: 3.0402 | P-Value: 2.6711e-03


An independent two-sample t-test was conducted to determine if the Fitbit and the ActiGraph report the same daily step measures. The test showed a t-statistic of 3.0402 and a p-value of 0.00267. Because the p-value is less than the α=0.05 threshold, we reject the null hypothesis. We conclude that there is a statistically significant difference between the step counts recorded by the two devices, the Fitbit does not report the same step measures as the ActiGraph.

### Weekend Warriors
Pick one of the devices: Are the subjects equally active across each day of the week? (as determined
by daily steps in day of week, ANOVA)

In [10]:
dailyFitbitSteps['date'] = pd.to_datetime(dailyFitbitSteps['date'])
dailyFitbitSteps['dayOfWeek'] = dailyFitbitSteps['date'].dt.dayofweek

daysData = []
for day in range(7):
    daySteps = dailyFitbitSteps[dailyFitbitSteps['dayOfWeek'] == day]['steps'].tolist()
    if len(daySteps) > 0:
        daysData.append(daySteps)

fStat, pValue = CustomANOVA(*daysData)
print(f"ANOVA F-Statistic: {fStat:.4f} | P-Value: {pValue:.4e}")

ANOVA F-Statistic: 5.8282 | P-Value: 2.1161e-05


A one-way ANOVA test was conducted to determine if there is a difference in the mean step counts depending on the day of the week. The test showed an F-Statistic of 5.8282 and a highly significant p-value of 0.000021161. Because the p-value is well below the standard α=0.05 threshold, we reject the null hypothesis. We can conclude that the subjects are not equally active across each day of the week, their physical activity levels change significantly depending on the day.

### Seasonality
In the two year data set (‘multiyear’ directory), you’ll find daily step totals. Across the two years,
were all months traveled equally? (repeated measures ANOVA)

In [11]:
multiyearDf['date'] = pd.to_datetime(multiyearDf['date'])
multiyearDf['month'] = multiyearDf['date'].dt.month
multiyearDf['year'] = multiyearDf['date'].dt.year

monthlyData = multiyearDf.groupby(['year', 'month'])['steps'].mean().reset_index()

uniqueYears = monthlyData['year'].unique()
monthsConditions = []

for month in range(1, 13):
    monthList = []
    
    for year in uniqueYears:
        yearMonthData = monthlyData[(monthlyData['year'] == year) & 
                                    (monthlyData['month'] == month)]['steps']
        
        if yearMonthData.empty:
            overallAvg = multiyearDf[multiyearDf['year'] == year]['steps'].mean()
            monthList.append(overallAvg)
        else:
            monthList.append(yearMonthData.values[0])
            
    monthsConditions.append(monthList)

fStat, pValue = RepeatedMeasuresANOVA(*monthsConditions)

print("--- Seasonality across Months (rANOVA) ---")
print(f"Repeated Measures F-Statistic: {fStat:.4f}")
print(f"Repeated Measures P-Value:     {pValue:.4e}")

--- Seasonality across Months (rANOVA) ---
Repeated Measures F-Statistic: 0.6180
Repeated Measures P-Value:     7.9437e-01


The Repeated Measures ANOVA resulted in a p-value of 0.794. Because this is greater than 0.05, there is no statistically significant difference in daily step totals across the different months, so we can conclude that all months were traveled equally and there is no significant evidence of seasonality in this dataset. :p

# --- END OF CODE ---